# Level 2 — Vision Transformers (ViT-S/16, 선택적으로 Swin-Tiny)

**목표**: ViT (와 선택적으로 Swin-T) 를 직접 구현하고 Multi-task 로 연결하여 Level 1 의 CNN 들과 비교합니다.

**Pretrained 가중치**: ImageNet `.pth` 파일을 본인이 구현한 모델의 `state_dict` 에 로드하는 것은 허용됩니다. 출처를 명시하세요. **`timm` / `torchvision.models` import 는 금지** 입니다.

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/Leem1nwon/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.vit import vit_small_patch16_224
# from src.models.swin import SwinTiny  # 선택 사항

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Colab `Run All` 재현 시 wandb 는 비활성화합니다 (조교 채점 환경 = WANDB_DISABLED=true).
# 본인이 직접 학습하며 로깅하려면 이 줄을 주석 처리하고 RUN_TRAINING=True, wandb.login() 을 사용하세요.
import os
os.environ["WANDB_DISABLED"] = "true"

# ===== 재현 모드 스위치 =====
# RUN_TRAINING = False → 학습을 건너뛰고 사전학습 체크포인트(.pth)를 로드해 eval/그림만 재현 (조교 채점 경로).
# RUN_TRAINING = True  → 처음부터 학습 (H100 권장; T4 에서는 매우 오래 걸립니다).
RUN_TRAINING = False

WANDB_PROJECT = "aue8088-pa2" if RUN_TRAINING else None   # 재현 모드에서는 wandb 끔
WANDB_TAGS    = ["level2"]

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# --- 사전학습 체크포인트 다운로드 (재현 모드 RUN_TRAINING=False) ----------------
# checkpoints/ 에 .pth 가 없으면 Google Drive 에서 체크포인트 zip 을 받아 압축 해제.
# (제출 전 체크포인트 zip 을 본인 드라이브에 올린 뒤 아래 ID 를 교체하세요.)
CKPT_DIR = "../checkpoints"
GDRIVE_CKPT_ID = "PUT_CHECKPOINT_ZIP_ID_HERE"   # ← 체크포인트 zip 의 Google Drive 파일 ID

if not RUN_TRAINING and not os.path.exists(f"{CKPT_DIR}/level2_vit_pretrained.pth"):
    if GDRIVE_CKPT_ID == "PUT_CHECKPOINT_ZIP_ID_HERE":
        print("[주의] GDRIVE_CKPT_ID 를 체크포인트 zip 파일 ID 로 교체하세요.")
    else:
        import gdown
        gdown.download(id=GDRIVE_CKPT_ID, output="../checkpoints.zip", quiet=False)
        with zipfile.ZipFile("../checkpoints.zip") as zf:
            zf.extractall("..")
        print("체크포인트 다운로드/해제 완료")
else:
    print(f"체크포인트 준비 상태: RUN_TRAINING={RUN_TRAINING}")

In [ ]:
# ImageNet pretrained 가중치를 본인 ViT 구현체에 로드하는 절차 (RUN_TRAINING=True 경로에서만 사용)
#
#   from scripts.vit_load_pretrained import load_pretrained_vit
#   model = vit_small_patch16_224()
#   load_pretrained_vit(model)   # timm 키 → 본인 state_dict 키 remap 후 strict=False 로드
#                                # Multi-task head 는 task 종속이므로 random init 유지.
#
# remap 규칙: blocks.{i}.mlp.fc1 → blocks.{i}.mlp.0, blocks.{i}.mlp.fc2 → blocks.{i}.mlp.3,
#            head.{weight,bias} 는 1000-class ImageNet 분류기라 drop.
# 사용한 체크포인트 출처(checkpoints/vit_s16_imagenet.pth)와 매칭된 키 개수는 리포트에 기재할 것.
#
# 재현 모드(RUN_TRAINING=False)에서는 이 절차가 필요 없습니다 — 학습 완료된
# level2_vit_pretrained.pth 를 그대로 load_state_dict 하므로 다음 셀이 처리합니다.
print("pretrained remap 절차는 다음 셀의 RUN_TRAINING=True 경로에서 실행됩니다.")

In [ ]:
from torch import nn

def load_vit(stem):
    """학습 완료된 ViT 체크포인트(.pth)를 로드 (재현 모드)."""
    m = vit_small_patch16_224().to(device)
    ck = torch.load(f"{CKPT_DIR}/{stem}.pth", map_location="cpu")
    m.load_state_dict(ck["state_dict"])
    m.to(device).eval()
    return m, ck.get("history")

def train_vit(stem, pretrained, epochs=25):
    """ViT 를 학습하고 체크포인트(fp32)와 history 를 저장한다."""
    set_seed(SEED, deterministic=True)
    model = vit_small_patch16_224().to(device)
    if pretrained:
        # ImageNet remap → 백본만 로드, head 는 random init 유지
        from scripts.vit_load_pretrained import load_pretrained_vit
        load_pretrained_vit(model)
    optim = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level2-{stem}",
        config={"backbone": "vit_s16", "pretrained": pretrained,
                "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED},
        tags=WANDB_TAGS + ["vit_s16", "pretrained" if pretrained else "scratch"],
    )
    trainer = MultiTaskTrainer(model, optim, sched, losses, device, TrainConfig(epochs=epochs), logger=logger)
    history = trainer.fit(train_loader, val_loader)
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", confusion_matrices(val_pred, val_tgt)[a], CLASS_NAMES[a])
    logger.finish()
    os.makedirs(CKPT_DIR, exist_ok=True)
    torch.save({"state_dict": model.float().state_dict(), "history": history,
                "seed": SEED, "pretrained": pretrained, "lr": 5e-4},
               f"{CKPT_DIR}/{stem}.pth")
    return model, history

# scratch vs pretrained — RUN_TRAINING 에 따라 학습 또는 체크포인트 로드
VIT_RUNS = {"level2_vit_scratch": False, "level2_vit_pretrained": True}
results = {}
for stem, pre in VIT_RUNS.items():
    model, history = (train_vit(stem, pre) if RUN_TRAINING else load_vit(stem))
    results[stem] = {"model": model, "history": history, "pretrained": pre}
    print(f"[{stem}] {'학습 완료' if RUN_TRAINING else '체크포인트 로드'}")

## 분석 (리포트 필수 포함 항목)

1. **CNN vs Transformer**: 동일 epoch 예산 하에서 ResNet-50 (Level 1) 과 ViT-S (Level 2) 의 Avg-MF1 을 비교하세요.
2. **Pretrained vs Scratch**: 약 5천 장 규모의 소규모 데이터셋에서 ImageNet 초기화가 실제로 얼마나 도움이 되는지 정량적으로 보고하세요.
3. **속성별 거동**: ViT 가 ResNet 대비 Weather 와 Time of Day 사이의 오류 분포를 다르게 가져가는지, 그 원인을 가설로 제시하세요.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.metrics import (
    average_macro_f1, per_attribute_macro_f1, per_class_prf,
    collect_predictions, confusion_matrices, CLASS_NAMES,
)

# 1) scratch vs pretrained — Avg-MF1 / 속성별 Macro-F1 (val)
print(f"{'run':22s} {'init':>10s} {'Avg-MF1':>8s}  weather  scene  timeofday")
preds_cache = {}
for stem, info in results.items():
    preds, _, tgts, _ = collect_predictions(info["model"], val_loader, device)
    preds_cache[stem] = (preds, tgts)
    avg = average_macro_f1(preds, tgts); per = per_attribute_macro_f1(preds, tgts)
    init = "pretrained" if info["pretrained"] else "scratch"
    print(f"{stem:22s} {init:>10s} {avg:8.4f}  {per['weather']:.3f}  {per['scene']:.3f}  {per['timeofday']:.3f}")

# best ViT = Avg-MF1 가 가장 높은 run
best_stem = max(preds_cache, key=lambda s: average_macro_f1(*preds_cache[s]))
print(f"\nbest ViT = {best_stem}")

# 2) best ViT 의 속성별 per-class P / R / F1
print(f"\n[{best_stem}] per-class P/R/F1")
bp, bt = preds_cache[best_stem]
for a in ATTRIBUTES:
    prf = per_class_prf(bp, bt)[a]
    print(f"  {a}:")
    for cls, p, r, f in zip(CLASS_NAMES[a], prf['precision'], prf['recall'], prf['f1']):
        print(f"    {cls:14s} P={p:.3f} R={r:.3f} F1={f:.3f}")

# 3) scratch vs pretrained 학습 곡선 (history 가 있을 때)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for stem, info in results.items():
    h = info["history"]; ls = "-" if info["pretrained"] else "--"
    if h and "train_loss" in h:
        ax[0].plot(h["train_loss"], ls=ls, label=stem)
    if h and "val_avg_mf1" in h:
        ax[1].plot(h["val_avg_mf1"], ls=ls, label=stem)
ax[0].set_title("Train loss (ViT scratch vs pretrained)"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
ax[1].set_title("Val Avg-MF1"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

# 4) best ViT 속성별 정규화 Confusion Matrix
cms = confusion_matrices(bp, bt)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax_, a in zip(axes, ATTRIBUTES):
    sns.heatmap(cms[a], annot=True, fmt=".2f", cmap="Blues", cbar=False,
                xticklabels=CLASS_NAMES[a], yticklabels=CLASS_NAMES[a], ax=ax_)
    ax_.set_title(f"{best_stem} — {a}"); ax_.set_xlabel("pred"); ax_.set_ylabel("true")
plt.tight_layout(); plt.show()